In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import pandas as pd

path = "UpdatedResumeDataSet.csv"  # Update this path as needed

df = pd.read_csv(path)

print(df.head())
print(df.columns)
print(df.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'UpdatedResumeDataSet.csv'

In [ ]:
# Explore the dataset

print("=" * 50)
print("DATASET SHAPE")
print("=" * 50)
print(df.shape)

print("\n" + "=" * 50)
print("COLUMN NAMES")
print("=" * 50)
print(df.columns)

print("\n" + "=" * 50)
print("FIRST 5 ROWS")
print("=" * 50)
print(df.head())

print("\n" + "=" * 50)
print("MISSING VALUES")
print("=" * 50)
print(df.isnull().sum())

print("\n" + "=" * 50)
print("NUMBER OF JOB CATEGORIES")
print("=" * 50)
print(df["Category"].nunique())

print("\n" + "=" * 50)
print("CATEGORY DISTRIBUTION")
print("=" * 50)
print(df["Category"].value_counts())

DATASET SHAPE
(962, 2)

COLUMN NAMES
Index(['Category', 'Resume'], dtype='object')

FIRST 5 ROWS
       Category                                             Resume
0  Data Science  Skills * Programming Languages: Python (pandas...
1  Data Science  Education Details \r\nMay 2013 to May 2017 B.E...
2  Data Science  Areas of Interest Deep Learning, Control Syste...
3  Data Science  Skills â¢ R â¢ Python â¢ SAP HANA â¢ Table...
4  Data Science  Education Details \r\n MCA   YMCAUST,  Faridab...

MISSING VALUES
Category    0
Resume      0
dtype: int64

NUMBER OF JOB CATEGORIES
25

CATEGORY DISTRIBUTION
Category
Java Developer               84
Testing                      70
DevOps Engineer              55
Python Developer             48
Web Designing                45
HR                           44
Hadoop                       42
Sales                        40
Data Science                 40
Mechanical Engineer          40
ETL Developer                40
Blockchain                   40

In [ ]:

import nltk
nltk.download('stopwords')

import re
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def clean_resume(text):

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', ' ', text)

    # Remove emails
    text = re.sub(r'\S+@\S+', ' ', text)

    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z ]', ' ', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text)

    # Remove stopwords
    words = text.split()
    words = [word for word in words if word not in stop_words]

    return " ".join(words)

# Create cleaned resume column
df["Cleaned_Resume"] = df["Resume"].apply(clean_resume)

print(df[["Category", "Cleaned_Resume"]].head())

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


       Category                                     Cleaned_Resume
0  Data Science  skills programming languages python pandas num...
1  Data Science  education details may may b e uit rgpv data sc...
2  Data Science  areas interest deep learning control system de...
3  Data Science  skills r python sap hana tableau sap hana sql ...
4  Data Science  education details mca ymcaust faridabad haryan...


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Convert text into numerical features

tfidf = TfidfVectorizer(
    max_features=5000
)

X = tfidf.fit_transform(df["Cleaned_Resume"])

y = df["Category"]

print("Feature Matrix Shape:", X.shape)
print("Target Shape:", y.shape)

Feature Matrix Shape: (962, 5000)
Target Shape: (962,)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Split dataset

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Create model

model = LogisticRegression(
    max_iter=1000
)

# Train model

model.fit(X_train, y_train)

# Predict

y_pred = model.predict(X_test)

# Accuracy

accuracy = accuracy_score(y_test, y_pred)

print("Training Samples:", X_train.shape[0])
print("Testing Samples :", X_test.shape[0])
print("Accuracy:", round(accuracy * 100, 2), "%")

Training Samples: 769
Testing Samples : 193
Accuracy: 99.48 %


In [ ]:
skills_db = [

    # Languages
    "python", "java", "c", "c++", "javascript",

    # Data Science
    "pandas", "numpy", "scipy", "matplotlib",
    "seaborn", "scikit learn", "machine learning",
    "deep learning", "data analysis",
    "data visualization", "nlp", "nltk",
    "opencv", "tensorflow", "keras",

    # Databases
    "sql", "mysql", "mongodb",
    "postgresql", "oracle",

    # Web
    "html", "css", "django", "flask",

    # DevOps
    "docker", "kubernetes",
    "aws", "jenkins",

    # Tools
    "git", "github",
    "tableau", "power bi",
    "excel",

    # Big Data
    "hadoop", "spark",

    # Testing
    "selenium",
    "automation testing",

    # Security
    "cyber security",
    "network security"
]

In [ ]:
import re

def extract_skills(text):

    text = text.lower()

    found_skills = []

    for skill in skills_db:

        pattern = r'\b' + re.escape(skill) + r'\b'

        if re.search(pattern, text):
            found_skills.append(skill)

    return sorted(list(set(found_skills)))

In [ ]:
role_skills = {

    "Data Science": [
        "python",
        "pandas",
        "numpy",
        "sql",
        "machine learning",
        "tableau"
    ],

    "Python Developer": [
        "python",
        "django",
        "flask",
        "sql",
        "git"
    ],

    "Java Developer": [
        "java",
        "sql",
        "git"
    ],

    "DevOps Engineer": [
        "docker",
        "kubernetes",
        "aws",
        "jenkins",
        "git"
    ],

    "Testing": [
        "selenium",
        "automation testing",
        "sql"
    ]
}

In [ ]:
def analyze_resume(predicted_role, detected_skills):

    required_skills = role_skills.get(predicted_role, [])

    missing_skills = []

    for skill in required_skills:
        if skill not in detected_skills:
            missing_skills.append(skill)

    if len(required_skills) > 0:
        score = (len(detected_skills) - len([s for s in detected_skills if s not in required_skills]))
        score = max(score, 0)
        score = (score / len(required_skills)) * 100
    else:
        score = 0

    return round(score, 2), missing_skills

In [ ]:
!pip install pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 81.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 74.7 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.


In [ ]:
import pdfplumber

def extract_text_from_pdf(pdf_path):

    text = ""

    with pdfplumber.open(pdf_path) as pdf:

        for page in pdf.pages:

            page_text = page.extract_text()

            if page_text:
                text += page_text + "\n"

    return text

In [ ]:
pdf_path = "sample_resume.pdf"  # Update this path as needed

resume_text = extract_text_from_pdf(pdf_path)

print(resume_text[:1000])

Jestina Mangol
Python Developer
Highly motivated Python Developer with 1 year of experience in developing robust and efficient software solutions.
Proficient in Python frameworks like Django and Flask, with a solid understanding of front-end technologies like HTML,
CSS, and JavaScript. Demonstrated ability to solve complex problems, excellent debugging skills, and a keen interest in
learning new technologies. Committed to delivering high-quality results within stipulated timelines.
jestina.mangol@gmail.com Employment History
(249) 346-3648 Senior Python Developer at Asurion, TN
Jul 2023 - Present
Nashville, TN
• Led the development and implementation of a new data processing
system using Python, improving data efficiency by 50% and
significantly reducing the time spent on manual data tasks.
Education • Successfully automated 5 key business processes using Python
scripts, resulting in a productivity increase of 30% and an annual
Bachelor of Science in
cost saving of $100,000.
Software E

In [ ]:
cleaned_resume = clean_resume(resume_text)

vector = tfidf.transform([cleaned_resume])

predicted_role = model.predict(vector)[0]

skills = extract_skills(resume_text)

score, missing_skills = analyze_resume(
    predicted_role,
    skills
)

print("Predicted Role:", predicted_role)
print()
print("Detected Skills:", skills)
print()
print("Resume Score:", score)
print()
print("Missing Skills:", missing_skills)

Predicted Role: Python Developer

Detected Skills: ['css', 'data analysis', 'django', 'flask', 'html', 'javascript', 'machine learning', 'numpy', 'pandas', 'python', 'scipy', 'tensorflow']

Resume Score: 60.0

Missing Skills: ['sql', 'git']


In [ ]:
def generate_suggestions(score, missing_skills):

    suggestions = []

    if score < 40:
        suggestions.append("Add more relevant technical skills.")
        suggestions.append("Include academic or personal projects.")
        suggestions.append("Improve alignment with the target role.")

    elif score < 70:
        suggestions.append("Consider adding more role-specific skills.")
        suggestions.append("Strengthen project descriptions.")

    else:
        suggestions.append("Good resume profile.")
        suggestions.append("Continue improving with advanced skills.")

    for skill in missing_skills:
        suggestions.append(f"Learn or add {skill}")

    return suggestions

In [ ]:
suggestions = generate_suggestions(
    score,
    missing_skills
)

print("Suggestions:")
for s in suggestions:
    print("-", s)

Suggestions:
- Consider adding more role-specific skills.
- Strengthen project descriptions.
- Learn or add sql
- Learn or add git


In [ ]:
import pickle

# Save model
with open("resume_model.pkl", "wb") as f:
    pickle.dump(model, f)

# Save vectorizer
with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)

print("Model and Vectorizer Saved Successfully!")

Model and Vectorizer Saved Successfully!


In [ ]:
with open("resume_model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

with open("tfidf_vectorizer.pkl", "rb") as f:
    loaded_tfidf = pickle.load(f)

print("Files Loaded Successfully!")

Files Loaded Successfully!


In [ ]:
def resume_analyzer(resume_text):

    # Predict Role
    cleaned_resume = clean_resume(resume_text)

    vector = loaded_tfidf.transform([cleaned_resume])

    predicted_role = loaded_model.predict(vector)[0]

    # Skills
    detected_skills = extract_skills(resume_text)

    # Score + Missing Skills
    score, missing_skills = analyze_resume(
        predicted_role,
        detected_skills
    )

    # Suggestions
    suggestions = generate_suggestions(
        score,
        missing_skills
    )

    return {
        "Predicted Role": predicted_role,
        "Detected Skills": detected_skills,
        "Resume Score": score,
        "Missing Skills": missing_skills,
        "Suggestions": suggestions
    }

In [ ]:
results = resume_analyzer(resume_text)

for key, value in results.items():
    print(f"\n{key}:")
    print(value)


Predicted Role:
Python Developer

Detected Skills:
['css', 'data analysis', 'django', 'flask', 'html', 'javascript', 'machine learning', 'numpy', 'pandas', 'python', 'scipy', 'tensorflow']

Resume Score:
60.0

Missing Skills:
['sql', 'git']

Suggestions:
['Consider adding more role-specific skills.', 'Strengthen project descriptions.', 'Learn or add sql', 'Learn or add git']


In [ ]:
!pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 67.1 MB/s eta 0:00:00
  Attempting uninstall: pillow
    Found existing installation: pillow 12.2.0
    Uninstalling pillow-12.2.0:
      Successfully uninstalled pillow-12.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pdfplumber 0.11.10 requires Pillow>=12.2.0, but you have pillow 11.3.0 which is incompatible.


In [ ]:
def analyze_pdf(pdf_file):

    text = extract_text_from_pdf(pdf_file)

    results = resume_analyzer(text)

    output = f"""
Predicted Role: {results['Predicted Role']}

Detected Skills:
{', '.join(results['Detected Skills'])}

Resume Score:
{results['Resume Score']}

Missing Skills:
{', '.join(results['Missing Skills'])}

Suggestions:
"""

    for suggestion in results["Suggestions"]:
        output += f"\n- {suggestion}"

    return output

In [ ]:
import gradio as gr

app = gr.Interface(
    fn=analyze_pdf,
    inputs=gr.File(label="Upload Resume PDF"),
    outputs=gr.Textbox(label="Analysis Result", lines=20),
    title="AI Resume Analyzer & Job Role Predictor",
    description="Upload a resume PDF to predict job role, detect skills, identify gaps, and get suggestions."
)

app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b47fa212bc505f0539.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
